## 🧠 YOLO-MultiSpectral: Demo Notebook

### 🔍 Section 1: Introduction

This notebook demonstrates how to modify a YOLO model 5 channel input.
Dataset weeds galore


---

### ⚙️ Step 1: Setup Environment

Below we install dependencies and clone the YOLO-MultiSpec repository.


In [2]:
!python --version

Python 3.10.18


In [ ]:
import os
#  Clone repository -- 
#  Only required for running from COLAB
run_from_colab = True
if run_from_colab:
    # Clone your GitHub repo
    !git clone https://github.com/aesparon/YOLO-Multispectral.git

    # Change directory to the repo root
    %cd YOLO-Multispectral

    # Now install requirements
    !pip install -r code/requirements.txt
else:
    # install reqiured packages on lacal 
    req_path = os.path.abspath(os.path.join("..", "..", "code", "requirements.txt"))
    !pip install -r "{req_path}"

# ✅ Install YOLO and cuda supprt
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [3]:
import os

current_folder = os.getcwd()
print("📁 Current folder:", current_folder)


📁 Current folder: d:\PD\Publications\Yolo_mod\github\YOLO-Multispectral\examples\notebooks


In [ ]:

# Check cuda support
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


### 🔍 Step 2. download weeds galore dataset

Download dataset

In [10]:
req_path = os.path.abspath(os.path.join("..", "..", "code"))
print(req_path)

d:\PD\Publications\Yolo_mod\github\YOLO-Multispectral\code


In [11]:

# add code path for script files
import sys 
# code folder
yolo_source_path = req_path #  '../code/'
sys.path.insert(1, yolo_source_path)

from utils_downloader import download_and_extract_zip

# Set parameters
url = "https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"
output_zip = "weedsgalore-dataset.zip"
extract_dir = "weedsgalore-dataset"

# Call the function
download_and_extract_zip(url, output_zip, extract_dir)

🔍 Checking file info...
📥 Downloading with resume support...


Downloading: 100%|██████████| 321M/321M [02:26<00:00, 2.30MB/s] 


✅ Download complete: weedsgalore-dataset.zip
📦 Extracting...


Extracting: 100%|██████████| 1119/1119 [00:01<00:00, 815.05file/s]

✅ Extracted to: weedsgalore-dataset


### 🔍 Section 3: Create train/val/test


In [ ]:
from pre_process_images import *

# Step 1.  Merge all date folder images into single folder  ###########################################################
input_folders = [
    "./weedsgalore-dataset/weedsgalore-dataset/2023-05-25/images/",
    "./weedsgalore-dataset/weedsgalore-dataset/2023-05-30/images/",
    "./weedsgalore-dataset/weedsgalore-dataset/2023-06-06/images/",
    "./weedsgalore-dataset/weedsgalore-dataset/2023-06-15/images/"
]
output_folder = "./weedsgalore-dataset/images_merge/"

merge_files(input_folders, output_folder)

In [ ]:

# Step 2.  Stack RGB images and convert from RGB uint16 to uint8   #######################################################3
input_folder_RGB = "./weedsgalore-dataset/images_merge/"
output_folder_RGB_uint16 = "D:/PD/Publications/Yolo_mod/github/YOLO-Multispectral/examples/notebooks/weedsgalore-dataset/RGB/RGB_uint16/"

# just run once
if not os.path.exists(output_folder_RGB_uint16):
    # Stack RGB bands
    batch_stack_rgb_triplets(input_folder_RGB, output_folder_RGB_uint16)

    output_folder_RGB_uint8_stretch = "./weedsgalore-dataset/RGB/RGB_uint8_stretch/"
    convert_folder_uint16_to_uint8(output_folder_RGB_uint16, output_folder_RGB_uint8_stretch, method='stretch')

    output_folder_RGB_uint8_norm = "./weedsgalore-dataset/RGB/RGB_uint8_normalize/"
    convert_folder_uint16_to_uint8(output_folder_RGB_uint16, output_folder_RGB_uint8_norm , method='normalize')


# Step 3.  Stack RGBRN images and convert from RGBRN uint16 to uint8   #######################################################3
input_folder_RGBRN = "./weedsgalore-dataset/images_merge/"
output_folder_RGBRN_uint16 = "./weedsgalore-dataset/RGBRN/RGBRN_uint16/"
if not os.path.exists(output_folder_RGBRN_uint16):
    # Stack RGBNR bands
    batch_stack_5band(input_folder_RGBRN, output_folder_RGBRN_uint16)

    output_folder_RGBRN_uint8_stretch = "./weedsgalore-dataset/RGBRN/RGBRN_uint8_stretch/"
    convert_folder_uint16_to_uint8(output_folder_RGBRN_uint16, output_folder_RGBRN_uint8_stretch, method='stretch')

    output_folder_RGBRN_uint8_norm = "./weedsgalore-dataset/RGBRN/RGBRN_uint8_normalize/"
    convert_folder_uint16_to_uint8(output_folder_RGBRN_uint16, output_folder_RGBRN_uint8_norm, method='normalize')

### Step 4 separate to train/val/test 

In [ ]:

train_val_test_list = ['train' , 'val', 'test']

for train_val_test in train_val_test_list: 

    #  RGB  ############################################################################################
    txt_split_file_to_sort = './weedsgalore-dataset/weedsgalore-dataset/splits/' + train_val_test + '.txt'
    input_instances_folder = './weedsgalore-dataset/RGB/RGB_uint8_normalize/'
    output_folder_instances = './weedsgalore-dataset/train_val_test/RGB/' + train_val_test + '/images/'

    #sort_images (txt_file_to_sort ,input_semantics_folder ,  output_folder_semantics)
    sort_images_from_split_files (txt_split_file_to_sort ,input_instances_folder ,  output_folder_instances)

    #  RGBRN  ############################################################################################
    txt_split_file_to_sort = './weedsgalore-dataset/weedsgalore-dataset/splits/' + train_val_test + '.txt'
    input_instances_folder = './weedsgalore-dataset/RGBRN/RGBRN_uint8_normalize/'
    output_folder_instances = './weedsgalore-dataset/train_val_test/RGBRN/' + train_val_test + '/images/'

    #sort_images (txt_file_to_sort ,input_semantics_folder ,  output_folder_semantics)
    sort_images_from_split_files (txt_split_file_to_sort ,input_instances_folder ,  output_folder_instances)


### Train model

In [10]:


# add hic fix - test and fix - then remove
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


train_data = 'weed_RGBRN' 

# parameters
num_epochs = 1000   # force stop by patience
batch_size = 16   # 4   # yolo_config['batch']
patience = 50
images_test_path = ''    # set if using test

data_base_path =  'D:/PD/yolo_mod/data/WeedsGalore/data_processed/train_val_test_V7/RGBNR/'
project_base_path = 'D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/'

##############################################   parameters   ##########################################################
# can auto detect
number_of_channels = 5   
clip_size= "600"
# ############################      uint16      ###################################################################
# ############################      uint8      ###################################################################
# expect folders for images/labels in train/val/test
images_train_path= data_base_path +  "train/" 
images_val_path=  data_base_path + "val/" 
# set to '' or None id NO test data
images_test_path=  data_base_path + "test/"     

classes_list = ['maize','amaranth','grass','quickweed','other']
number_of_classes = len(classes_list)

model_base =   'yolo11x-seg.yaml'    # 'yolo11n-seg.pt'
folder = model_base[0:6]
#Initialise additional channels when more then 3 (RGB)
channel_init_mode = 'random'       # 'avg'   'zeros'     'random'    'same'
# trans_learn = True     # False      # True       #False    #True

use_cbam=False
use_eca=False
use_spectral=False 
use_dropblock=False
drop_prob=0.1

In [11]:
# create train yaml
def create_train_yaml(images_train_path, images_val_path ,classes_list , number_of_channels, output_yaml ,  images_test_path = None):
    ###    CREATE data - YAML file for training    #######################################################################
    yaml_content = "train: " + images_train_path + "\n"
    yaml_content =yaml_content +  "val: " + images_val_path + "\n"
    if not (images_test_path == '' or images_test_path == None):
        yaml_content =yaml_content +  "test: " + images_test_path + "\n"

    # extra channels
    yaml_content =yaml_content +  "nc: " + str ( len(classes_list) ) + "\n"
    yaml_content =yaml_content +  "channels: " + str( number_of_channels) + "\n"
    #yaml_content =yaml_content  + cat_yaml_str     #"names: ['tree']"

    #classes_list = ['maize','amaranth','grass','quickweed','other']  
    # create name classes for yolo training
    names_str = "names: [" + ",".join(classes_list) + "]"
    print(names_str)
    yaml_content =yaml_content +  names_str + "\n"

    #output_yaml =  project_base_path + "data_" + train_data + ".yaml"
    

    with Path(output_yaml).open('w') as f:
        f.write(yaml_content) 
    ##########################################################################################################################



In [12]:
output_yaml =  project_base_path + "data_" + train_data + ".yaml"
    
create_train_yaml(images_train_path, images_val_path ,classes_list , number_of_channels, output_yaml ,  images_test_path = None)

names: [maize,amaranth,grass,quickweed,other]


In [ ]:

from mod_pt_model_seg import *


model_base_str = model_base.replace('.','_') # for file path - remove .extension
output_model_train_base =  project_base_path  +  train_data + '_' + model_base_str + '_method_' + channel_init_mode + '_eps_' + str(num_epochs) 

# from mod_pt_model_seg_v1.py
model_created ,output_model_train  = patch_yolo_seg_ckpt(
    #input_ckpt='C:/Users/andre/AppData/Roaming/QGIS/QGIS3/profiles/default/python/plugins/aigist/QGIS_DL/models_base/YOLO8/yolo11x-seg.pt',
    model_base=model_base,
    output_model_train= output_model_train_base,
    in_channels=number_of_channels,
    nc=number_of_classes,
    #trans_learn = trans_learn,
    channel_init_mode=channel_init_mode,
    use_cbam=use_cbam, 
    use_eca=use_eca, 
    use_spectral=use_spectral, 
    use_dropblock=use_dropblock, 
    drop_prob=0.1
)


📦 Loading pretrained YOLO segmentation model: yolo11x-seg.yaml
🛠️ Patching input conv from 3 to 5 using 'random'
🧠 Updating segmentation head to 5 classes
Saving patched model to: D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg/mod_model/yolo11x-seg.yaml_no_TL.pt


In [15]:

# use created model   #################################################
model_path =os.path.dirname(model_created)
model_file = os.path.basename(model_created)

#  Check later - cannot pass path to   -> model = YOLO( path + 'yolo11x.pt') 
if model_path != '':
    os.chdir(model_path)
model = YOLO( model_file )


# Automatically use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

##########################################  TRAIN  #######################################
results = model.train(
        #task='segment',
        batch= batch_size,   # batch_size,    # 32 on small    set to -1 to optimise to GPU
        data=output_yaml,
        epochs= num_epochs ,      #num_epochs,
        imgsz=int(clip_size),
        device=device,
        patience =patience ,
        #conf = conf,
        name = output_model_train,
        exist_ok=True,
)  


New https://pypi.org/project/ultralytics/8.3.160 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.148  Python-3.10.18 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/data_weed_RGBRN.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=600, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11x-seg.yaml_no_TL.pt, momentum=0.937, mosa

train: Scanning D:\PD\yolo_mod\data\WeedsGalore\data_processed\train_val_test_V7\RGBNR\train\labels.cache... 104 images, 0 backgrounds, 0 corrupt: 100%|██████████| 104/104 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.00.0 ms, read: 3117.0122.2 MB/s, size: 1758.0 KB)


val: Scanning D:\PD\yolo_mod\data\WeedsGalore\data_processed\train_val_test_V7\RGBNR\val\labels.cache... 26 images, 0 backgrounds, 0 corrupt: 100%|██████████| 26/26 [00:00<?, ?it/s]


Plotting labels to D:\PD\yolo_mod\working2\projects\weeds_galore\RGBRN_tests_final2\weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001111, momentum=0.9) with parameter groups 176 weight(decay=0.0), 187 weight(decay=0.0005), 186 bias(decay=0.0)
Image sizes 608 train, 608 val
Using 8 dataloader workers
Logging results to D:\PD\yolo_mod\working2\projects\weeds_galore\RGBRN_tests_final2\weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg
Starting training for 1000 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


...

                   all         26       2376          0          0          0          0          0          0          0          0



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


    12/1000      18.5G       3.05      3.246      1.896      2.181       1067        608: 100%|██████████| 7/7 [00:01<00:00,  3.66it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  4.49it/s]

                   all         26       2376          0          0          0          0          0          0          0          0


KeyboardInterrupt: 

In [26]:
results.save_dir

WindowsPath('D:/PD/yolo_mod/working2/projects/weeds_galore/RGBRN_tests_final2/weed_RGBRN_yolo11x-seg_yaml_method_random_eps_1000_no_TL_seg')

In [32]:

try:
    # can be incremented on 2nd run
    output_model_train =results.save_dir
except NameError:
    output_model_train = output_model_train

if images_test_path != '':
    # run test evaluation if exists
    model_save_path = str(output_model_train)
    best_last = 'best'
    eval_best_last(model_save_path , output_yaml , best_last)

    best_last = 'last'
    eval_best_last(model_save_path , output_yaml , best_last)

Ultralytics 8.3.148  Python-3.10.18 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLO11x-seg summary (fused): 203 layers, 62,009,631 parameters, 0 gradients, 318.9 GFLOPs


FileNotFoundError: [34m[1mval: [0mError loading data from None
See https://docs.ultralytics.com/datasets for dataset formatting guidance.